# 01 - Dataset et preprocessing

Objectif : vérifier le dataset synthétique, les images, les labels et préparer les fonctions de prétraitement.

Fichiers concernés : `data/synthetic_cases.csv`, `data/sample_images/`, `src/preprocessing.py`.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
SAMPLE_IMAGES_DIR = DATA_DIR / "sample_images"
SRC_DIR = PROJECT_ROOT / "src"
API_DIR = PROJECT_ROOT / "api"
APP_DIR = PROJECT_ROOT / "app"
EVAL_DIR = PROJECT_ROOT / "eval"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
PROMPTS_DIR = PROJECT_ROOT / "prompts"
TESTS_DIR = PROJECT_ROOT / "tests"
OUTPUTS_DIR.mkdir(exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("PROJECT_ROOT =", PROJECT_ROOT)

In [ ]:
import pandas as pd
from pathlib import Path

df = pd.read_csv(DATA_DIR / "synthetic_cases.csv")
display(df.head())
required_repo = {"case_id", "image_path", "source", "label", "split", "quality", "notes"}
required_prompt = {"case_id", "filename", "expected_label"}
print("Colonnes dépôt OK:", required_repo <= set(df.columns))
print("Colonnes consigne filename/expected_label OK:", required_prompt <= set(df.columns))
print("Labels invalides:", sorted(set(df["label"]) - {"normal", "suspected_opacity", "uncertain"}))
df["resolved_image_path"] = df["image_path"].apply(lambda p: PROJECT_ROOT / p)
df["image_exists"] = df["resolved_image_path"].apply(lambda p: p.exists())
display(df[["case_id", "image_path", "label", "quality", "image_exists"]].head())

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

sample = df.groupby("label", group_keys=False).head(2).head(6)
fig, axes = plt.subplots(1, len(sample), figsize=(3 * len(sample), 3))
if len(sample) == 1:
    axes = [axes]
for ax, (_, row) in zip(axes, sample.iterrows()):
    ax.imshow(Image.open(row["resolved_image_path"]), cmap="gray")
    ax.set_title(f"{row['case_id']}\n{row['label']}")
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
try:
    from src.preprocessing import load_image, basic_quality_flag
    img = load_image(df.loc[0, "resolved_image_path"])
    print("load_image OK:", img.size, img.mode)
    print("basic_quality_flag:", basic_quality_flag(df.loc[0, "resolved_image_path"]))
except Exception as exc:
    print("src/preprocessing.py à compléter:", repr(exc))

def get_image_info(image_path):
    with Image.open(image_path) as img:
        return {"width": img.width, "height": img.height, "mode": img.mode, "format": img.format}

def assess_image_quality(image_path):
    name = Path(image_path).name.lower()
    return "limited" if "uncertain" in name or "limited" in name else "good"

def preprocess_image(image_path, size=(512, 512)):
    return Image.open(image_path).convert("RGB").resize(size)

In [ ]:
summary = df.groupby(["label", "quality", "split"]).size().reset_index(name="n")
out = OUTPUTS_DIR / "dataset_summary.csv"
summary.to_csv(out, index=False)
print(out)
display(summary)

Conclusion : le dataset est réutilisable. Le code final doit documenter l'écart `image_path/label` vs `filename/expected_label`.